In [ ]:
# from gulps.synthesis_plugin import GulpsSynthesisPlugin
import numpy as np
from gulps.viz.polytope_viz import _plot_coverage_set
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import (
    CXGate,
    RZXGate,
    UGate,
    UnitaryGate,
    XXPlusYYGate,
    iSwapGate,
)
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator, average_gate_fidelity
from qiskit.quantum_info.random import random_unitary
from qiskit.transpiler import (
    InstructionProperties,
    PassManager,
    Target,
    generate_preset_pass_manager,
)
from qiskit.transpiler.passes import Optimize1qGatesDecomposition
from tqdm import tqdm
from weylchamber import c1c2c3

from gulps.synthesis.gulps_decomposer import GulpsDecomposer
from gulps.synthesis_pass import GulpsDecompositionPass
from gulps.core.invariants import GateInvariants
from gulps import logger

# logger.setLevel("DEBUG")
logger.setLevel("INFO")

In [23]:
from qiskit.synthesis.two_qubit.xx_decompose import XXDecomposer

N = 100
slope, offset = 1e-10, 1e-12

basis_fidelity = {
    strength: 1.0 - (slope * strength / (np.pi / 2) + offset)
    for strength in [
        # np.pi / 2,
        # np.pi / 4,
        # np.pi / 6,
        # np.pi / 8,
        np.pi / 10,
        np.pi / 12,
        np.pi / 16,
        np.pi / 24,
    ]
}
xx_decomposer = XXDecomposer(basis_fidelity)
fidelities = []
for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)
    v = Operator(xx_decomposer(u))
    fid = average_gate_fidelity(u, v)
    fidelities.append(fid)

    if fid < 1 - 1e-6:
        print(f"Unitary {idx} fidelity is low: {fid:.8f}")
        print("Canonical invariants:")
        print("U:", c1c2c3(u))
        print("V:", c1c2c3(v))
        print("\n")

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:16<00:00,  6.09it/s]


In [24]:
gate_set = [
    # CXGate(),
    # CXGate().power(1 / 2),
    # CXGate().power(1 / 3),
    # CXGate().power(1 / 4),
    CXGate().power(1 / 5),
    CXGate().power(1 / 6),
    CXGate().power(1 / 8),
    CXGate().power(1 / 12),
]
costs = [1 - v for v in basis_fidelity.values()]
decomposer = GulpsDecomposer(gate_set, costs, precompute_polytopes=True)
# if hasattr(decomposer.isa, "coverage_set"):
#     _plot_coverage_set(decomposer.isa.coverage_set)

In [25]:
fidelities = []

for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)
    v = Operator(decomposer(u))
    fid = average_gate_fidelity(u, v)
    fidelities.append(fid)

    if fid < 1 - 1e-6:
        print(f"Unitary {idx} fidelity is low: {fid:.8f}")
        print("Canonical invariants:")
        print("U:", c1c2c3(u))
        print("V:", c1c2c3(v))
        print("\n")
        continue

# Summary statistics
fidelities = np.array(fidelities)
print(f"\nSummary across {len(fidelities)} samples:")
print(f"  Median fidelity: {np.median(fidelities)}")
print(f"  Mean fidelity:   {np.mean(fidelities)}")
print(f"  Minimum fidelity:{np.min(fidelities)}")

  0%|          | 0/100 [00:00<?, ?it/s]

  1%|          | 1/100 [00:00<00:41,  2.41it/s]


QiskitError: 'TwoQubitWeylDecomposition: failed to diagonalize M2. Please report this at https://github.com/Qiskit/qiskit-terra/issues/4159. Input: [[Complex { re: -0.11157666573800933, im: 0.7845330226143421 }, Complex { re: 0.2376208562962207, im: -0.19156282520559317 }, Complex { re: -0.42958839392188175, im: 0.04266138969976216 }, Complex { re: -0.24818900159193769, im: -0.1758824433191899 }],\n [Complex { re: -0.2980787392656578, im: 0.1687197975686463 }, Complex { re: 0.18796338225758813, im: -0.4007692338327353 }, Complex { re: 0.7419564686188614, im: 0.038646488926146985 }, Complex { re: -0.2849584578198614, im: 0.23139189297604076 }],\n [Complex { re: -0.29463112906394306, im: 0.3444057181202351 }, Complex { re: -0.5509052492956521, im: 0.5506827732348055 }, Complex { re: 0.11629134978310188, im: 0.316547060980162 }, Complex { re: -0.06547932057424405, im: 0.2642268483724215 }],\n [Complex { re: 0.02337722291995678, im: 0.22084379945480515 }, Complex { re: -0.24748279666865589

In [ ]:
# # %reload_ext  snakeviz
# %load_ext snakeviz
# import cProfile

# u = random_unitary(4, seed=0)
# cProfile.run("decomposer._run(u)", "profile_timings/temph.prof")